# 26 — Fine-tuning de BETO sobre el título y la garantía

**Idea.** Reemplazar o complementar `prob_texto_new` (regresión logística sobre TF-IDF del título + garantía, 0,8414 sola) con un **transformer pre-entrenado en español, ajustado para `new` vs `used`**.

**Material:**
- AP, `material/redes predictivo/Intro a Redes para datos estructurados.pdf`: neurona y red *fully connected* (slides 5-7), backpropagation (8), *cross-entropy* (9), descenso por gradiente, mini-batch y épocas (10-14), *learning rate* (16), **Adam** (19-21), dropout (24) y **Transformers** (slide 25: solo el diagrama de la arquitectura).
- SIA, `material/redes neuronales/S12 - Redes Neuronales - NLP.pdf` y `S12_Redes_Neuronales_NLP.ipynb`: tokens y vocabulario (11), bolsa de palabras + regresión logística (12), embeddings (13-17, 20), **atención y embeddings contextuales** (26-28), el transformer (29), pre-entrenamiento como *transfer learning* (35). La notebook usa un DistilBERT **ya ajustado por otros** y muestra que la bolsa de palabras + regresión logística es difícil de superar.
- **No está en el material:** hacer el fine-tuning nosotros (loop de entrenamiento, AdamW con *weight decay*, *scheduler* lineal con *warmup*), BETO, el tokenizador WordPiece y la cabeza de clasificación sobre `[CLS]`. Slide 29 de S12: *"La arquitectura de Transformers va más allá de los contenidos de este curso"*.

**Fine-tuning y por qué es distinto de P9.**
- En P9 (Fase G) se usaron embeddings **congelados** de un modelo entrenado para **paráfrasis**. Ese modelo ya había decidido que lo importante de una frase es el **tema**: "iPhone nuevo" e "iPhone usado" daban casi el mismo vector (0,82). Es lo que S12 muestra con *good*/*bad* en word2vec.
- Con **fine-tuning**, se parte de BETO (BERT pre-entrenado con texto en español), se le agrega una capa de clasificación sobre `[CLS]` y se **siguen entrenando todos los pesos** con nuestras etiquetas (backpropagation + Adam). Las representaciones se reacomodan para la tarea: "nuevo", "sin uso", "a estrenar" de un lado; "usado", "detalles" del otro. Y con atención, cada palabra se lee en contexto ("sin uso" ≠ "uso").

**Configuración** (`bert_texto.py`): `dccuchile/bert-base-spanish-wwm-cased`, entrada `[CLS] título [SEP] garantía [SEP]`, `max_length` 64, 2 épocas, AdamW con lr 2e-5 y *weight decay* 0,01, *warmup* lineal el primer 10% de los pasos y después baja lineal a 0, batch 32, `device="mps"`. Un ajuste sobre 56.000 filas: **22,6 min** (20,4 de entrenamiento).

**Cross-fitting sin leakage (15 modelos en lugar de 30).**
- **5 "globales"** `sin_k`: entrenados sin el fold k (56.000 filas). Predicen el fold k y el test.
- **10 "de a pares"** `sin_j_k`: entrenados sin los folds j y k (42.000 filas). Predicen j y k.
- **Para el fold externo k:**
  - fila de **validación** (fold k) → `sin_k`, que no la vio;
  - fila de **train** del fold j ≠ k → `sin_j_k`, que no vio **ni esa fila ni la validación**.

  Así ninguna probabilidad de entrenamiento sale de un modelo que vio la validación. Si se reusaran las 5 globales en todos los folds, la feature de train del fold j vendría de `sin_j`, que **sí** entrenó con el fold k de validación: un sesgo chico pero real.
- **Desigualdad conocida:** la feature de train sale de modelos de 42.000 filas y la de validación de modelos de 56.000, igual que en `PrediccionOOF`.
- **En la entrega:** train = OOF de los 5 globales; test = promedio de los 5 globales en el test. No hace falta reentrenar.

In [1]:
import sys
sys.path.append("..")
import os, time, warnings
import numpy as np
import pandas as pd
from joblib import Memory, Parallel, delayed
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict

from features import (aplanar_anidadas, agregar_features_contenido, agregar_categoricas_crudas, agregar_combos,
                      PrediccionOOF, armar_modelo_texto, armar_pipeline_histgb_combos, armar_pipeline_final, INDICES_CAT)

SEED = 42
crudo = pd.read_json("../data/train_data.jsonlines", lines=True)
y = (crudo["condition"] == "new").astype(int)
y_np = y.to_numpy()
train = agregar_combos(agregar_categoricas_crudas(agregar_features_contenido(aplanar_anidadas(crudo))))
folds = np.empty(len(train), dtype=int)
for k, (_, v) in enumerate(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(train, y)):
    folds[v] = k + 1
carpeta_lab = ".cache_laboratorio"
carpeta_bert = os.path.join(carpeta_lab, "bert")
memoria_texto = Memory(location=carpeta_lab, verbose=0)

oof_ref = pd.read_csv(os.path.join(carpeta_lab, "oof_p7c_duracion.csv"))
assert (oof_ref["fold"].to_numpy() == folds).all()


def acc_por_fold(prob):
    return pd.Series((np.asarray(prob) >= 0.5).astype(int) == y_np).groupby(folds).mean().to_numpy()


def comparar(acc, ref):
    diferencias = np.round(acc - ref, 10)
    gana = int((diferencias > 0).sum())
    return {"media": acc.mean(), "dif_media": diferencias.mean(), "dif_desvio": diferencias.std(), "folds_gana": gana,
            "pasa": gana == 5 or (gana == 4 and diferencias.mean() > diferencias.std())}


acc_hgb = acc_por_fold(oof_ref["histgb"])
acc_final = acc_por_fold(0.5 * oof_ref["catboost"] + 0.5 * oof_ref["histgb"])

# Probabilidades de BETO guardadas por bert_texto.py
n = len(train)
bert_global = np.full(n, np.nan)            # OOF de los 5 globales (fila del fold k <- sin_k)
bert_test = []                               # predicciones de los 5 globales en el test
minutos = {}
for k in range(1, 6):
    d = np.load(os.path.join(carpeta_bert, f"sin_{k}.npz"))
    es_train = d["filas"] < n
    bert_global[d["filas"][es_train]] = d["prob"][es_train]
    bert_test.append(pd.Series(d["prob"][~es_train], index=d["filas"][~es_train] - n).sort_index().to_numpy())
    minutos[f"sin_{k}"] = float(d["minutos"])
bert_pares = {}                              # (j, k) -> probabilidades sobre todas las filas de train (NaN fuera de j y k)
hay_pares = all(os.path.exists(os.path.join(carpeta_bert, f"sin_{j}_{k}.npz")) for j in range(1, 6) for k in range(j + 1, 6))
for j in range(1, 6) if hay_pares else []:
    for k in range(j + 1, 6):
        d = np.load(os.path.join(carpeta_bert, f"sin_{j}_{k}.npz"))
        p = np.full(n, np.nan)
        p[d["filas"]] = d["prob"]
        bert_pares[(j, k)] = bert_pares[(k, j)] = p
        minutos[f"sin_{j}_{k}"] = float(d["minutos"])
assert not np.isnan(bert_global).any()
print(f"Modelos cargados: {len(minutos)} (de a pares: {'sí' if hay_pares else 'todavía no'}). "
      f"Minutos de entrenamiento por modelo: " + ", ".join(f"{c} {v:.1f}" for c, v in minutos.items()))

Modelos cargados: 15 (de a pares: sí). Minutos de entrenamiento por modelo: sin_1 20.4, sin_2 18.3, sin_3 18.7, sin_4 21.8, sin_5 18.8, sin_1_2 13.8, sin_1_3 14.0, sin_1_4 14.1, sin_1_5 14.0, sin_2_3 13.7, sin_2_4 14.2, sin_2_5 14.1, sin_3_4 13.9, sin_3_5 13.8, sin_4_5 13.9


## (a) El modelo de texto solo: BETO contra la regresión logística actual

Las dos con la misma CV: la logística (`armar_modelo_texto()`, título + garantía) con `cross_val_predict`, y BETO con las OOF de los 5 globales. También se mide el promedio de los dos y su correlación: si se equivocan en filas distintas, BETO puede servir como **complemento** aunque no le gane.

In [2]:
archivo_log = os.path.join(carpeta_lab, "oof_texto_logistica.csv")
if os.path.exists(archivo_log):
    prob_log = pd.read_csv(archivo_log)["prob"].to_numpy()
else:
    X_texto = pd.DataFrame({"title": crudo["title"], "warranty_texto": crudo["warranty"].fillna("")})
    prob_log = cross_val_predict(armar_modelo_texto(), X_texto, y, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                                 method="predict_proba", n_jobs=5)[:, 1]
    pd.DataFrame({"prob": prob_log}).to_csv(archivo_log, index=False)

texto_solo = pd.DataFrame({
    "logística (actual)": acc_por_fold(prob_log),
    "BETO": acc_por_fold(bert_global),
    "promedio logística + BETO": acc_por_fold((prob_log + bert_global) / 2),
}, index=[f"fold {k}" for k in range(1, 6)])
texto_solo.loc["media"] = texto_solo.mean()
print(texto_solo.round(4))
print("\nBETO vs logística:", {k: round(float(v), 4) for k, v in comparar(acc_por_fold(bert_global), acc_por_fold(prob_log)).items()})
print("Correlación:", np.corrcoef(prob_log, bert_global)[0, 1].round(3),
      "| desacuerdo:", ((prob_log >= 0.5) != (bert_global >= 0.5)).mean().round(3))

        logística (actual)    BETO  promedio logística + BETO
fold 1              0.8421  0.8469                     0.8594
fold 2              0.8406  0.8506                     0.8584
fold 3              0.8436  0.8491                     0.8584
fold 4              0.8382  0.8503                     0.8594
fold 5              0.8427  0.8519                     0.8592
media               0.8414  0.8497                     0.8589

BETO vs logística: {'media': 0.8497, 'dif_media': 0.0083, 'dif_desvio': 0.0027, 'folds_gana': 5.0, 'pasa': 1.0}
Correlación: 0.889 | desacuerdo: 0.117


## (b) `prob_bert_new` en el HistGB P7a con duración

Dos variantes:
- **reemplazo:** `prob_bert_new` en lugar de `prob_texto_new` (el TF-IDF 500 se mantiene);
- **extra:** las dos.

Comparación pareada contra el HistGB P7a con duración (0,9412). Primero las funciones; después el filtro rápido y la medición exacta. En la exacta la columna cambia en cada fold externo, así que se ajusta fold por fold (en paralelo) en vez de usar `cross_validate`.

In [3]:
def columna_bert(k):
    col = bert_global.copy()                  # validación (fold k): sin_k
    for j in range(1, 6):
        if j != k:
            filas_j = folds == j
            col[filas_j] = bert_pares[(j, k)][filas_j]   # train del fold j: sin_j_k
    return col


def con_texto_cacheado(pipeline):
    transformers = pipeline.named_steps["preprocesador"].transformers
    i = [nm for nm, _, _ in transformers].index("texto")
    transformers[i] = ("texto", Pipeline([("oof", PrediccionOOF(armar_modelo_texto(), random_state=SEED)),
                                          ("identidad", "passthrough")], memory=memoria_texto), ["title", "warranty_texto"])
    return pipeline


def armar(modelo, variante):
    pipeline = con_texto_cacheado(armar_pipeline_histgb_combos() if modelo == "histgb" else armar_pipeline_final())
    transformers = pipeline.named_steps["preprocesador"].transformers
    if variante == "reemplazo":
        i = [nm for nm, _, _ in transformers].index("texto")
        transformers[i] = ("texto", "passthrough", ["prob_bert_new"])     # misma posición: INDICES_CAT no cambia
    elif variante == "extra":
        transformers.append(("bert", "passthrough", ["prob_bert_new"]))  # al final: INDICES_CAT no cambia
    return pipeline


def correr_fold(k, modelo, variante):
    X = train.copy()
    X["prob_bert_new"] = columna_bert(k)
    tr, va = folds != k, folds == k
    params = {"modelo__cat_features": INDICES_CAT} if modelo == "catboost" else {}
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        pipeline = armar(modelo, variante).fit(X[tr], y[tr], **params)
        return k, pipeline.predict_proba(X[va])[:, 1]


def oof(modelo, variante):
    prob = np.empty(n)
    for k, p in Parallel(n_jobs=5)(delayed(correr_fold)(k, modelo, variante) for k in range(1, 6)):
        prob[folds == k] = p
    return prob

## Filtro rápido (con un sesgo chico): las OOF de los 5 globales en todos los folds

Antes de entrenar los 10 modelos "sin dos folds" (~3 h), un filtro barato: la columna `prob_bert_new` es **la misma en todos los folds**, la OOF de los 5 globales (cada fila de train recibe la predicción del BETO que no la vio).

**El sesgo:** para el fold externo k, una fila de train del fold j ≠ k recibe la predicción de `sin_j`, que **sí aprendió con el fold k** (la validación). El modelo de etapa 2 se entrena con una feature que "vio" un poco las etiquetas de la validación. Debería ser chico (cada BETO ve 56.000 filas), pero **va a favor de BETO**. Por eso:
- **si ni así suma, se descarta** (el sesgo solo podía ayudarlo);
- **si suma, se confirma con el esquema exacto** (los 10 "sin dos folds") antes de creerle.

In [4]:
from sklearn.model_selection import cross_validate
resultados, probs = {}, {}
X_filtro = train.assign(prob_bert_new=bert_global)
for variante in ["reemplazo", "extra"]:
    inicio = time.time()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        acc = cross_validate(armar("histgb", variante), X_filtro, y, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                             scoring="accuracy", n_jobs=5)["test_score"]
    resultados[f"FILTRO (sesgado) HistGB P7a, BETO {variante}"] = {**comparar(acc, acc_hgb), "minutos": (time.time() - inicio) / 60}
    print(variante, {k: round(float(v), 4) for k, v in resultados[f"FILTRO (sesgado) HistGB P7a, BETO {variante}"].items()})
pd.DataFrame(resultados).T.round(4)

reemplazo {'media': 0.941, 'dif_media': -0.0002, 'dif_desvio': 0.0009, 'folds_gana': 2.0, 'pasa': 0.0, 'minutos': 0.5671}


extra {'media': 0.9424, 'dif_media': 0.0012, 'dif_desvio': 0.0009, 'folds_gana': 4.0, 'pasa': 1.0, 'minutos': 0.5665}


,media,dif_media,dif_desvio,folds_gana,pasa,minutos
"FILTRO (sesgado) HistGB P7a, BETO reemplazo",0.941,-0.000243,0.000864,2,False,0.567143
"FILTRO (sesgado) HistGB P7a, BETO extra",0.942414,0.001171,0.000915,4,True,0.566469


## Medición exacta (cross-fitting con los 10 modelos "sin dos folds")

Se corre solo si existen los 10 modelos. Para cada fold externo k: validación ← `sin_k`; train del fold j ← `sin_j_k`.

In [5]:
for variante in ["reemplazo", "extra"] if hay_pares else []:
    inicio = time.time()
    probs[("histgb", variante)] = oof("histgb", variante)
    resultados[f"HistGB P7a, BETO {variante}"] = {**comparar(acc_por_fold(probs[("histgb", variante)]), acc_hgb),
                                                  "minutos": (time.time() - inicio) / 60}
    print(variante, resultados[f"HistGB P7a, BETO {variante}"])
pd.DataFrame(resultados).T.round(4)

reemplazo {'media': np.float64(0.9415000000000001), 'dif_media': np.float64(0.00025714284), 'dif_desvio': np.float64(0.0013398141646344475), 'folds_gana': 3, 'pasa': False, 'minutos': 0.5723320285479228}


extra {'media': np.float64(0.9425571428571429), 'dif_media': np.float64(0.0013142857000000001), 'dif_desvio': np.float64(0.001485027326485329), 'folds_gana': 4, 'pasa': np.False_, 'minutos': 0.6611717820167542}


,media,dif_media,dif_desvio,folds_gana,pasa,minutos
"FILTRO (sesgado) HistGB P7a, BETO reemplazo",0.941,-0.000243,0.000864,2,False,0.567143
"FILTRO (sesgado) HistGB P7a, BETO extra",0.942414,0.001171,0.000915,4,True,0.566469
"HistGB P7a, BETO reemplazo",0.9415,0.000257,0.00134,3,False,0.572332
"HistGB P7a, BETO extra",0.942557,0.001314,0.001485,4,False,0.661172


## Confirmación en el promedio final (CatBoost + HistGB + duración), semilla 42

Solo para las variantes que pasaron en el HistGB. La misma columna se suma a CatBoost (reemplazo: en la posición de `prob_texto_new`; extra: al final), se promedian 50/50 y se compara contra el modelo final (0,9420). La semilla 7 necesita otros 15 modelos de BETO (otros folds): se lanza solo si esto pasa.

In [6]:
pasan = [v for v in ["reemplazo", "extra"] if hay_pares and resultados[f"HistGB P7a, BETO {v}"]["pasa"]]
print("Pasan en el HistGB:", pasan or "ninguna")
for variante in pasan:
    inicio = time.time()
    probs[("catboost", variante)] = oof("catboost", variante)
    prom = 0.5 * probs[("catboost", variante)] + 0.5 * probs[("histgb", variante)]
    resultados[f"CatBoost, BETO {variante} (vs CatBoost final)"] = comparar(acc_por_fold(probs[("catboost", variante)]),
                                                                           acc_por_fold(oof_ref["catboost"]))
    resultados[f"Promedio final, BETO {variante} (vs 0,9420)"] = {**comparar(acc_por_fold(prom), acc_final),
                                                                  "minutos": (time.time() - inicio) / 60}
pd.DataFrame({"fold": folds, "y": y_np, "bert_oof_global": bert_global, "logistica_oof": prob_log,
              **{f"{m}_{v}": p for (m, v), p in probs.items()}}).to_csv(os.path.join(carpeta_lab, "oof_bert.csv"), index=False)
pd.DataFrame({"bert_test_promedio": np.mean(bert_test, axis=0)}).to_csv(os.path.join(carpeta_lab, "test_bert.csv"), index=False)
pd.DataFrame(resultados).T.round(4)

Pasan en el HistGB: ninguna


,media,dif_media,dif_desvio,folds_gana,pasa,minutos
"FILTRO (sesgado) HistGB P7a, BETO reemplazo",0.941,-0.000243,0.000864,2,False,0.567143
"FILTRO (sesgado) HistGB P7a, BETO extra",0.942414,0.001171,0.000915,4,True,0.566469
"HistGB P7a, BETO reemplazo",0.9415,0.000257,0.00134,3,False,0.572332
"HistGB P7a, BETO extra",0.942557,0.001314,0.001485,4,False,0.661172


## Conclusión

| Prueba | Accuracy | Contra | Diferencia (folds) |
|---|---|---|---|
| Texto solo: BETO | 0,8497 | logística 0,8414 | **+0,0083 (5/5)** |
| Texto solo: promedio logística + BETO | 0,8589 | logística | +0,0175 |
| Filtro sesgado: HistGB P7a + BETO extra | 0,9424 | 0,9412 | +0,0012 (4/5, pasa) |
| **Exacto: HistGB P7a + BETO extra** | **0,9426** | 0,9412 | **+0,0013 (4/5), desvío 0,0015 → no pasa** |
| Exacto: HistGB P7a con BETO en lugar de `prob_texto_new` | 0,9415 | 0,9412 | +0,0003 (3/5) |

Diferencias por fold (extra, exacto): −0,0009, +0,0016, +0,0015, +0,0037, +0,0006.

- **BETO es mejor modelo de texto que la logística** (5/5) y se equivoca en otras filas (correlación 0,889; no coinciden en el 11,7%). El fine-tuning sí aprende la tarea, a diferencia de los embeddings congelados de P9 (0,7247).
- **Dentro del modelo completo no alcanza el criterio:** +0,0013 en 4/5 folds, con una ganancia despareja (el fold 4 aporta casi un tercio y el fold 1 pierde). Cambia el 1,5% de las predicciones del HistGB (correlación 0,994). Interpretación (no verificada): la mayoría de las filas que BETO "arregla" respecto de la logística ya las resolvían el vendedor y las demás features.
- **Como reemplazo no sirve:** la ganancia de BETO aparece solo junto con la logística.
- Costo: 15 ajustes de BETO, ~4,4 h de entrenamiento en MPS (20 min por modelo de 56.000 filas y 14 min por modelo de 42.000).
- **Decisión:** no pasa, así que no se mide en el promedio final ni con la semilla 7, y no hay submission. Modelo final sin cambios (P7c + duración, 0,9420). Probabilidades guardadas en `.cache_laboratorio/bert/` (los 15 modelos), `oof_bert.csv` y `test_bert.csv` (promedio de los 5 globales en el test).